In [38]:
import pandas as pd
import numpy as np
import warnings,os,joblib
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import TimeSeriesSplit ,cross_validate
from sklearn.linear_model import LogisticRegression,LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor,StackingRegressor,VotingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score,make_scorer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.feature_selection import SelectKBest,f_regression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

pd.set_option('display.max_columns',None)


In [39]:
second=pd.read_csv(r'../data/processed/second_innings_clean.csv',parse_dates=['date'])
second.head()


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder,season,date,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,batting_team_wins
0,335982,2,RCB,KKR,1,1,R Dravid,AB Dinda,W Jaffer,1,0,1,NaN,0,NaN,NaN,NaN,2007/08,2008-04-18,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,0
1,335982,2,RCB,KKR,1,2,W Jaffer,AB Dinda,R Dravid,0,1,1,wides,0,NaN,NaN,NaN,2007/08,2008-04-18,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,0
2,335982,2,RCB,KKR,1,3,W Jaffer,AB Dinda,R Dravid,0,0,0,NaN,0,NaN,NaN,NaN,2007/08,2008-04-18,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,0
3,335982,2,RCB,KKR,1,4,W Jaffer,AB Dinda,R Dravid,1,0,1,NaN,0,NaN,NaN,NaN,2007/08,2008-04-18,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,0
4,335982,2,RCB,KKR,1,5,R Dravid,AB Dinda,W Jaffer,1,0,1,NaN,0,NaN,NaN,NaN,2007/08,2008-04-18,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,0


In [40]:
second['season_yr']=second['date'].dt.year.astype(int)

In [41]:
second['season_yr'].unique()

array([2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,
       2019, 2020, 2021, 2022, 2023, 2024])

In [42]:
second.isnull().sum()

match_id                  0
inning                    0
batting_team              0
bowling_team              0
over                      0
ball                      0
batter                    0
bowler                    0
non_striker               0
batsman_runs              0
extra_runs                0
total_runs                0
extras_type          116065
is_wicket                 0
player_dismissed     116560
dismissal_kind       116560
fielder              118306
season                    0
date                      0
venue                     0
team1                     0
team2                     0
toss_winner               0
toss_decision             0
winner                    0
result                    0
result_margin             0
target_runs               0
batting_team_wins         0
season_yr                 0
dtype: int64

In [43]:
match_totals=second.groupby('match_id')['total_runs'].sum().reset_index()
match_totals.rename(columns={'total_runs':'final_chase_score'},inplace=True)

match_df=second[['match_id','date','season_yr','venue','batting_team','bowling_team','target_runs','batting_team_wins']].drop_duplicates()

chase_df=pd.merge(match_df,match_totals,on='match_id',how='left')




In [44]:
chase_df.columns

Index(['match_id', 'date', 'season_yr', 'venue', 'batting_team',
       'bowling_team', 'target_runs', 'batting_team_wins',
       'final_chase_score'],
      dtype='object')

In [45]:
# Mela ulla cell-ah (Cell 3) run pannitu, idhai run pannu machi!

# 1. Powerplay stats create panrom
pp_df = second[second['over']<=6]

pp_stats = pp_df.groupby('match_id').agg(
    pp_runs=('total_runs','sum'),
    pp_wickets=('is_wicket','sum')
).reset_index()

pp_df['is_boundary'] = pp_df['batsman_runs'].apply(lambda x: 1 if x in [4,6] else 0)
pp_boundaries = pp_df.groupby('match_id')['is_boundary'].sum().reset_index().rename(columns={'is_boundary':'pp_boundaries'})

# 2. Merge panrom (Kandippa 'left' join thaan podanum)
chase_df = pd.merge(chase_df, pp_stats, on='match_id', how='left')
chase_df = pd.merge(chase_df, pp_boundaries, on='match_id', how='left')

# 3. Ippo dropna pannu (Error varathu)
chase_df.dropna(subset=['pp_runs'], inplace=True)

# 4. Matha features calculate panrom
chase_df['pp_crr'] = round((chase_df['pp_runs']/6), 2)
chase_df['runs_needed_after_pp'] = chase_df['target_runs'] - chase_df['pp_runs']
chase_df['rrr_after_pp'] = round((chase_df['runs_needed_after_pp']/14), 2)
chase_df['pressure_index'] = chase_df['rrr_after_pp'] * (chase_df['pp_wickets']+1)

chase_df[['target_runs','pp_runs','pp_wickets','rrr_after_pp','pressure_index']].head()

,target_runs,pp_runs,pp_wickets,rrr_after_pp,pressure_index
0,223.0,26,4,14.07,70.35
1,241.0,63,1,12.71,25.42
2,130.0,55,1,5.36,10.72
3,166.0,40,1,9.00,18.00
4,111.0,26,3,6.07,24.28


In [46]:
chase_df['is_impact_era']=chase_df['season_yr'].apply(lambda x: 1 if x >= 2023 else 0)
chase_df['is_t20_boom_era']=chase_df['season_yr'].apply(lambda x: 1 if x>=2018 else 0)

def get_venue_chase_avg(venue,match_df,df):
    past_matches=df[(df['venue']==venue) & (df['date']<match_df)]
    if len(past_matches)<3:
        return df[df['date']<match_df]['final_chase_score'].mean()
    return round(past_matches['final_chase_score'].mean(),2)

def get_team_recent_form(team,match_date,df):
    past_matches=df[(df['batting_team']==team) & (df['date']<match_date)].sort_values(by='date').tail(5)
    if len(past_matches)<2:
        return df[df['date']< match_date]['final_chase_score'].mean()
    return round(past_matches['final_chase_score'].mean(),2)


chase_df['venue_chase_avg']=chase_df.apply(lambda x: get_venue_chase_avg(x['venue'],x['date'],chase_df),axis=1)
chase_df['team_recent_chase_form']=chase_df.apply(lambda x: get_team_recent_form(x['batting_team'],x['date'],chase_df),axis=1)
    
global_mean=chase_df['final_chase_score'].mean()

chase_df['venue_chase_avg'].fillna(global_mean,inplace=True)
chase_df['team_recent_chase_form'].fillna(global_mean,inplace=True)


In [47]:
chase_df=chase_df.sort_values('date').reset_index(drop=True)

In [48]:
train_season=[2019,2020,2021,2022,2023]
test_season=[2024]

In [49]:
features_cols=['target_runs', 'is_impact_era', 'is_t20_boom_era',
    # Powerplay Dynamics
    'pp_runs', 'pp_wickets', 'pp_boundaries',
    # Chase Pressure Dynamics
    'pp_crr', 'rrr_after_pp', 'pressure_index',
    # Historical Context
    'venue_chase_avg', 'team_recent_chase_form',
    # Categorical
    'batting_team', 'bowling_team', 'venue',
    # Split feature
    'season_yr']

train_df=chase_df[chase_df['season_yr'].isin(train_season)].sort_values('date')
test_df=chase_df[chase_df['season_yr'].isin(test_season)].sort_values('date')


In [50]:
x_train=train_df[features_cols].reset_index(drop=True)
x_test=test_df[features_cols].reset_index(drop=True)
y_train=train_df['final_chase_score'].reset_index(drop=True)
y_test=test_df['final_chase_score'].reset_index(drop=True)

In [51]:
print(f' training season : {train_season}')
print(f' test_season: {test_season}')
print(f' x_train shape: {x_train.shape} range {y_train.min():.0f}- {y_train.max():.0f}')
print(f' x_test shape : {x_test.shape} y range {y_test.min():.0f}- {y_test.max():.0f}')
print(f' train row {len(x_train)}')

baseline_mae=mean_absolute_error(y_test,np.full(len(y_test),y_train.mean()))
baseline_rmse=mean_squared_error(y_test,np.full(len(y_test),y_train.mean()))**0.5

print(f' baseline mean pred Mae: {baseline_mae:.1f}')
print(f' baseline mean pred rmse :  {baseline_rmse:.1f}')
print(f' target mae: < {baseline_mae*0.5:.1f}')

 training season : [2019, 2020, 2021, 2022, 2023]
 test_season: [2024]
 x_train shape: (317, 15) range 59- 226
 x_test shape : (71, 15) y range 92- 262
 train row 317
 baseline mean pred Mae: 29.7
 baseline mean pred rmse :  37.7
 target mae: < 14.8


In [52]:
cat_cols=['batting_team', 
    'bowling_team', 
    'venue']

num_cols = [
    'target_runs', 
    'pp_runs', 
    'pp_wickets', 
    'pp_boundaries',
    'pp_crr', 
    'rrr_after_pp', 
    'pressure_index',
    'venue_chase_avg', 
    'team_recent_chase_form'
]

pass_cols = [
    'is_impact_era', 
    'is_t20_boom_era',
    'season_yr' 
]

assert set (cat_cols + num_cols+pass_cols)==set(features_cols),\
    f' mismatch missing : {set(features_cols)-set(cat_cols + num_cols +pass_cols)}'





In [53]:
preprocessor=ColumnTransformer(
    transformers=[
        ('cat',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))
        ]),cat_cols),
        ('num',Pipeline([
            ('imputer',SimpleImputer(strategy='median')),
            ('scaling',StandardScaler())
        ]),num_cols),
        ('pass','passthrough',pass_cols),

    ],remainder='drop')

In [54]:
models={
    'lg':LinearRegression(),
    'rf':RandomForestRegressor(n_estimators=300,max_depth=3,min_samples_leaf=10,max_features='sqrt',random_state=42,n_jobs=-1),
    'xgb':XGBRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,min_child_weight=15,reg_alpha=1.0,reg_lambda=5.0,random_state=42),
    'lgbm': LGBMRegressor(n_estimators=300,max_depth=8,min_samples_leaves=10,subsample=8,min_samples=15,reg_alpha=1.0,reg_lambda=5.0,random_state=42)

}

In [55]:
tscv=TimeSeriesSplit(n_splits=5)

def cv_mae_plain(pipeline,x,y):
    scores=[]
    for tr,val in tscv.split(x):
        pipeline.fit(x.iloc[tr],y.iloc[tr])
        scores.append(mean_absolute_error(y.iloc[val],pipeline.predict(x.iloc[val])))
    return np.mean(scores)

def cv_mae_early(model_cls,model_params,preproc,x,y,es_rounds=20):
    scores=[]
    for tr,val in tscv.split(x):
        x_tr,y_tr=x.iloc[tr],y.iloc[tr]
        x_val,y_val=x.iloc[val],y.iloc[val]

        preproc_fit=preproc.__class(**preproc.get_params())
        pipe=Pipeline([
            ('preprocessor',preprocessor),
            ('model',model_cls(**model_params))
        ])

    
        pipe.fit(
            x_tr,y_tr,m__eval_set =[(pipe.named_steps['preprocessor'].fit_transform(x_val),y_val)]\
            if False else None,

        )

        scores.append(mean_absolute_error(y_val,pipe.predict(x_val)))

    return np.mean(scores)

def xgb_objective(trail):
    params=dict(
        n_estimators=trail.suggest_int('n_estimators',100,600),
        max_depth=trail.suggest_int('max_depth',2,6),
        learning_rate=trail.suggest_float('lr',0.005,0.1,log=True),
        subsample=trail.suggest_float('subsample',0.5,0.9),
        colsample_bytree=trail.suggest_float('colsample_bytree',0.4,0.9),
        min_child_weight=trail.suggest_int('min_child_weight',10,50),
        reg_alpha=trail.suggest_float('reg_alpha',1.0,10.0),
        reg_lambda=trail.suggest_float('reg_lambda',1.0,15.0),
        gamma=trail.suggest_float('gamma',0.0,5.0),
        random_state=42,n_jobs=-1,verbosity=0,
    )

    pipe=Pipeline([('preprocessor',preprocessor),
                   ('model',XGBRegressor(**params))])
    return cv_mae_plain(pipe,x_train,y_train)

def lgbm_objective(trial):
    params=dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 600),
        max_depth         = trial.suggest_int('max_depth', 2, 6),
        learning_rate     = trial.suggest_float('lr', 0.005, 0.1, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 10, 40),   # small — anti overfit
        subsample         = trial.suggest_float('subsample', 0.5, 0.9),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 80),  # strong
        reg_alpha         = trial.suggest_float('reg_alpha', 1.0, 10.0),
        reg_lambda        = trial.suggest_float('reg_lambda', 1.0, 15.0),
        min_gain_to_split = trial.suggest_float('min_gain_to_split', 0.0, 2.0),  # new
        random_state      = 42, n_jobs=-1, verbose=-1,
    )


    pipe=Pipeline([
        ('preprocessor',preprocessor),
        ('model',LGBMRegressor(**params))
    ])

    return cv_mae_plain(pipe,x_train,y_train)


xgb_study=optuna.create_study(direction='minimize',pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
xgb_study.optimize(xgb_objective,n_trials=50,show_progress_bar=True)
best_xgb_p=xgb_study.best_params.copy()
best_xgb_p['learning_rate']=best_xgb_p.pop('lr')
best_xgb_p.update({'random_state':42,'n_jobs':-1,'verbosity':0})
print(f' best xgb cv mae : {xgb_study.best_value:.2f} runs')

print('optuna tune for lgbm')

lgbm_study=optuna.create_study(direction='minimize',pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
lgbm_study.optimize(lgbm_objective,n_trials=50,show_progress_bar=True)
best_lgbm_p=lgbm_study.best_params.copy()
best_lgbm_p['learning_rate']=best_lgbm_p.pop('lr')
best_lgbm_p.update({'random_state':42,'n_jobs':-1,'verbose':-1})
print(f' best lgbm cv mae : {lgbm_study.best_value:.2f} runs')

print(f'tuning the results')
print(f'xgb params : {best_xgb_p}')
print(f' lgbm params {best_lgbm_p}')


        

  0%|          | 0/50 [00:00<?, ?it/s]

 best xgb cv mae : 15.14 runs
optuna tune for lgbm


  0%|          | 0/50 [00:00<?, ?it/s]

 best lgbm cv mae : 15.53 runs
tuning the results
xgb params : {'n_estimators': 342, 'max_depth': 5, 'subsample': 0.803779850224797, 'colsample_bytree': 0.7167447138566932, 'min_child_weight': 10, 'reg_alpha': 5.7784702643029195, 'reg_lambda': 9.335839986786574, 'gamma': 4.762425032337346, 'learning_rate': 0.015136455147276238, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
 lgbm params {'n_estimators': 417, 'max_depth': 6, 'num_leaves': 19, 'subsample': 0.6751334050041796, 'min_child_samples': 20, 'reg_alpha': 4.993460441218143, 'reg_lambda': 10.430999069011532, 'min_gain_to_split': 0.7703661206293295, 'learning_rate': 0.01228959975715595, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


In [56]:
xgb_model_pipe=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',XGBRegressor(**best_xgb_p))
])

lgbm_model_pipe=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',LGBMRegressor(**best_lgbm_p))
])

lgbm_model_pipe.fit(x_train,y_train)
xgb_model_pipe.fit(x_train,y_train)

lgbm_train_pred=lgbm_model_pipe.predict(x_train)
xgb_train_pred=xgb_model_pipe.predict(x_train)
lgbm_pred=lgbm_model_pipe.predict(x_test)
xgb_pred=xgb_model_pipe.predict(x_test)

print(f'\n lgbm train mae: {mean_absolute_error(y_train,lgbm_train_pred):.2f}')
print(f'xgb train mae: {mean_absolute_error(y_train,xgb_train_pred):.2f}')





 lgbm train mae: 9.80
xgb train mae: 8.88


In [57]:
#ensemble
tuned_xgb=XGBRegressor(**best_xgb_p)
tuned_lgbm=LGBMRegressor(**best_lgbm_p)

tuned_rf=RandomForestRegressor(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=10,
    max_features=0.6,
    random_state=42,
    n_jobs=-1,
)

from sklearn.model_selection import KFold
stacking_cv=KFold(n_splits=5,shuffle=False)

stacker=StackingRegressor(
    estimators=[
        ('xgb',tuned_xgb),
        ('lgbm',tuned_lgbm),
        ('rf',tuned_rf),

    ],
    final_estimator=Ridge(alpha=10.0),
    cv=stacking_cv,
    passthrough=False,
    n_jobs=-1,
)

final_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('stacker',stacker),
])

print('fitting to the ensemble pipeline')
final_pipeline.fit(x_train,y_train)

train_preds_quick=final_pipeline.predict(x_train)
print(f' train mae:  {mean_absolute_error(y_train,train_preds_quick):.2f}')

fitting to the ensemble pipeline
 train mae:  8.99


In [58]:
from sklearn.metrics import mean_absolute_percentage_error

In [59]:
#cross validation

cv_results=cross_validate(
    xgb_model_pipe,x_train,y_train,cv=tscv,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1,

)

cv_train_maes=-cv_results['train_score']
cv_val_maes=-cv_results['test_score']

mean_cv_train=cv_train_maes.mean()
mean_cv_val=cv_val_maes.mean()
std_cv_val=cv_val_maes.std()


print(f'\n fold-wise val mae: {np.round(cv_val_maes,1)}')
print(f'cv train mae: {mean_cv_train:.2f} runs')
print (f' cv val mae : {mean_cv_val:.2f} runs +_ {std_cv_val:.2f}')
print(f' cv overfit gap : {mean_cv_val - mean_cv_train:.2f}')


#stability checkng
gap=mean_cv_val - mean_cv_train
if cv_train_maes.mean() <3 and mean_cv_val >15:
    print(f' severe overfitting')
elif gap > 8:
    print(f'mild overfitting')
elif mean_cv_val > baseline_mae * 0.75:
    print(f'underfitting')
elif gap <6 and std_cv_val < 5:
    print(f'good fit ')
else: 
    print(f' acceptable fit')


print(f'\n retraining the full x_train and final evaluation')
final_pipeline.fit(x_train,y_train)

y_train_pred=final_pipeline.predict(x_train)
train_mae=mean_absolute_error(y_train,y_train_pred)
train_rmse=mean_squared_error(y_train,y_train_pred)**0.5

print(f' final train mae: {train_mae:.1f}')
print(f' final train rmse: {train_rmse:.1f}')

mean_cv_mae=mean_cv_val
std_cv_mae=std_cv_val


 fold-wise val mae: [20.8 13.1 13.1 15.  14.1]
cv train mae: 8.79 runs
 cv val mae : 15.20 runs +_ 2.87
 cv overfit gap : 6.41
 acceptable fit

 retraining the full x_train and final evaluation
 final train mae: 9.0
 final train rmse: 13.2


In [60]:
# test evaluation

y_pred=xgb_model_pipe.predict(x_test)

test_mae=mean_absolute_error(y_test,y_pred)
test_rmse=mean_squared_error(y_test,y_pred)**0.5

test_r2=r2_score(y_test,y_pred)
test_mape=mean_absolute_percentage_error(y_test,y_pred)*100.0

within_10=(np.abs(y_test-y_pred)<=10).mean()*100.0
within_15=(np.abs(y_test-y_pred)<=15).mean()*100.0
within_20=(np.abs(y_test-y_pred)<=20).mean()*100.0
improvement=(1-test_mae/baseline_mae)*100.0

print(f' test evaluation results and predictins')

print(f'baseline mae : {baseline_mae:.1f}')
print(f'model test mae: {test_mae:.2f}')
print(f'model test rmse: {test_rmse:.2f}')
print(f'model test r2 : {test_r2:.4f}')
print(f'mape: {test_mape:.2f}%')
print(f' improvement vs baseline: {improvement:.2f}%')

print()
print(f'within +-10 runs: {within_10:.2f}%')
print(f'within +-15 runs: {within_15:.1f}%')
print(f'within +-20 runs: {within_20:.1f}%')

print(f' \n bias-variance summary')
print(f' train_mae: {train_mae:.1f}')
print(f' cv mae: {mean_cv_mae:.1f} +- {std_cv_mae:.1f}')
print(f' test mae : {test_mae:.1f}')
overfit_flag=mean_cv_mae - train_mae
print(f"gap (cv -train): {overfit_flag:.1f} {'good fit' if overfit_flag<6 else 'overfitting'}")

compare_df=pd.DataFrame({
    'actual(2024)':y_test.values[:10],
    'predicted(2024)':np.round(y_pred[:10]).astype(int),
    'abs error':np.abs(y_test.values[:10]-y_pred[:10]).round(1)

})
print(f'\n sample prediction vs actual')
print(compare_df.to_string(index=False))

 test evaluation results and predictins
baseline mae : 29.7
model test mae: 17.57
model test rmse: 23.84
model test r2 : 0.4716
mape: 9.96%
 improvement vs baseline: 40.82%

within +-10 runs: 38.03%
within +-15 runs: 52.1%
within +-20 runs: 69.0%
 
 bias-variance summary
 train_mae: 9.0
 cv mae: 15.2 +- 2.9
 test mae : 17.6
gap (cv -train): 6.2 overfitting

 sample prediction vs actual
 actual(2024)  predicted(2024)  abs error
          176              178        1.5
          204              188       16.5
          177              158       18.7
          162              153        8.8
          173              160       13.2
          178              155       22.9
          143              166       23.3
          246              198       48.4
          173              156       17.0
          186              172       14.1


In [61]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
labels=['baseline\n(mean)','train\nMae','cv\nmae','test\nmae']
values=[baseline_mae,train_mae,mean_cv_mae,test_mae]

colors=['#888780', '#1D9E75', '#378ADD', '#D85A30']
bars=axes[0].bar(labels,values,color=colors,width=0.5,edgecolor='white')
for bar,val in zip(bars,values):
    axes[0].text(bar.get_x() + bar.get_width()/2,bar.get_height()  + 0.3,
               f'{val:.1f}',ha='center',va='bottom',fontsize=11,fontweight='bold')
axes[0].axhline(12,color='red',linestyle='--',alpha=0.7,label='target : 12 runs')
axes[0].set_ylabel('mae ')
axes[0].set_title('overfit chek baseline -> train -> cv -> test')
axes[0].legend()

axes[1].scatter(y_test,y_pred,alpha=0.5,color='blue',s=20,label='predictions')
mn,mx=min(y_test.min(),y_pred.min()),max(y_test.max(),y_pred.max())
axes[1].plot([mn,mx],[mn,mx],'r--',linewidth=1.5,label='perfect')
axes[1].set_xlabel('actual total')
axes[1].set_ylabel('predicted score')
axes[1].set_title(f' actual vs predicted (r2={test_r2:.3f})')
axes[1].legend()

residuals=y_test.values -y_pred
axes[2].hist(residuals,bins=25,color='purple',edgecolor='white')
axes[2].axhline(0,color='red',linestyle='--',linewidth=1.5,label='zero error')
axes[2].axvline(residuals.mean(),color='lime',linestyle='--',label=f'mean error: {residuals.mean():.1f}')
axes[2].set_xlabel('residuals (actual - predicted)')
axes[2].set_ylabel('count')
axes[2].set_title('residuals distribution')
axes[2].legend()

plt.show()
    

In [62]:
# learning curve
from sklearn.model_selection import learning_curve

train_sizes,train_scores,val_scores=learning_curve(
    final_pipeline,x_train,y_train,cv=tscv,
    train_sizes=np.linspace(0.2,1.0,6),
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)

train_mae_lc=-train_scores.mean(axis=1)
val_mae_lc=-val_scores.mean(axis=1)
val_std_lc=val_scores.std(axis=1)

fig,ax=plt.subplots(figsize=(8,6))
ax.plot(train_sizes,train_mae_lc,'o-',color='green',label='train mae')
ax.plot(train_sizes,val_mae_lc,'o-',color='red',label='cv mae')
ax.fill_between(train_sizes,val_mae_lc-val_std_lc,val_mae_lc+val_std_lc,alpha=0.15,color='red')
ax.axhline(12,color='red',linestyle='--',alpha=0.5,label='target : 12 runs')
ax.set_xlabel('training samples')
ax.set_ylabel('mae(runs)')
ax.set_title('learning curve ')
ax.legend()
plt.tight_layout()
plt.show()